# Import Packages

In [13]:
# Imports

from pdf2image import convert_from_path
from PIL import Image
import csv
import os
import re

# Input Variables

In [14]:
# Input Variables

PDF_PATH = "input/nov_2025_characters.pdf" # Path to your PDF
NAMES_FILE = "input/nov_2025_character_list.txt" # Path to your page→name text file
OUTPUT_DIR = "output/" # Folder where JPEGs will be saved
DPI = 300 # Resolution for rasterizing (150–300 is typical)
JPEG_QUALITY = 95  # JPEG quality 1–95

In [ ]:
# margins for the card images
# the left and right margin are the same for both cards
left_margin = 377
right_margin = 2171

# top and bottom margin for the healthy card
top_margin_healthy = 555
bottom_margin_healthy = 1751

# top adn bottom margin for the injured card
top_margin_injured = 1815
bottom_margin_injured = 3015

# create box to crop card
crop_healthy = (left_margin,top_margin_healthy,right_margin,bottom_margin_healthy)
crop_injured = (left_margin,top_margin_injured,right_margin,bottom_margin_injured)

# Functions

In [17]:
def parse_names_file(
        path: str, 
        delimiter: str = '\t', 
        skip_rows: int = 0
        ) -> dict[int, str]:
    """Returns page number and character name.
    
    This function parses a text document looking for 2 fields on each line,
    the first with a page number and the second with the name of the character.

    Args:
        path (str): Path to the text file to parse.
        delimiter (str): Character to use to determine column 1 and column 2.
        skip_rows (int): Number of rows to skip at start of the file.

    Returns:
        names: A dictionary containing the page number and the name
        of the character on that page.

    """

    names = {}

    with open(path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file):
            if line_number < skip_rows:
                continue

            page_str, character = line.strip().split(delimiter,maxsplit=1)
            names[int(page_str)] = character

    return names

In [18]:
def character_name_to_filename (
    character_name: str
    ) -> str:
    """Returns a safe name to use when saving the file.
    
    This function replaces any non-safe characters in the input character name
    with an underscore.

    Args:
        character_name (str): Name of the character to save.

    Returns:
        filename: A string containing an OS safe file name to use.

    """
    # Replace invalid characters with underscore
    filename = re.sub(r'[<>:"/\\|?*\x00-\x1f]', '_', character_name)
    
    # Collapse multiple underscores/spaces
    filename = re.sub(r'[_\s]+', '_', filename)
    
    # Strip leading/trailing underscores and dots
    filename = filename.strip('_.')
    return filename

In [19]:
def crop_card(
    page_image: Image.Image,
    character_name: str,
    output_path: str,
    healthy_crop_box: tuple,
    injured_crop_box: tuple
    ) -> str:
    """Returns success or failure message.
    
    This function crops the healthy and injureed character cards and saves
    them to output directory as a jpeg.

    Args:
        page_image (image): Image of PDF page with the charcter cards.
        character_name (str): Name of the character.
        output_path (str): Path to save card images.
        healthy_crop_box (tuple): Coordinates used to crop the healthy charcter card.
        injured_crop_box (tuple): Coordinates used to crop the injured charcter card.

    Returns:
        crop_message: A string indicating if the images were cropped or if it failed.
    """

    # clean the character name to be os file name friendly
    character_filename = character_name_to_filename(character_name)

    # create full path and filename
    healthy_file = os.path.join(output_path, f"{character_filename}_healthy.jpg")
    injured_file = os.path.join(output_path, f"{character_filename}_injured.jpg")

    # crop the two images from the pdf page
    healthy = page_image.crop(healthy_crop_box)
    injured = page_image.crop(injured_crop_box)

    # Convert to rgb (pdf can be rgba and we don't want the alpha channel)
    if healthy.mode != "RGB":
        healthy = healthy.convert("RGB")

    if injured.mode != "RGB":
        injured = injured.convert("RGB")

    crop_message = character_name + ": "

    # save files
    try:
        healthy.save(healthy_file, "JPEG", quality=JPEG_QUALITY)
        crop_message = crop_message + 'healthy succeeded'
    except:
        crop_message = crop_message + 'healthy failed'

    try:
        injured.save(injured_file, "JPEG", quality=JPEG_QUALITY)
        crop_message = crop_message + ' and injured succeeded.'
    except:
        crop_message = crop_message + ' and injured failed.'

    return crop_message


# Process Files

In [23]:
# read in the page number and character names
card_list = parse_names_file(NAMES_FILE)

# get the page numbers to loop
page_numbers = sorted(card_list.keys())

# print start message
print(f"Processing {len(page_numbers)} characters \n")

# loop through the pages
for page_num in page_numbers:
    character_name = card_list[page_num]

    # rasterize the page
    try:
        pages = convert_from_path(
            PDF_PATH,
            dpi=DPI,
            first_page=page_num,
            last_page=page_num
        )
    except:
        print(f"    ⚠ Could not render {character_name} on page {page_num}, skipping.")

    crop_status = crop_card(
        page_image=pages[0],
        character_name=character_name,
        output_path=OUTPUT_DIR,
        healthy_crop_box=crop_healthy,
        injured_crop_box=crop_injured
    )

    print(crop_status)

print("\n Processing complete.")

Processing 13 characters 

Dormammu: healthy succeeded and injured succeeded.
Cassandra Nova: healthy succeeded and injured succeeded.
Sentinel MK4: healthy succeeded and injured succeeded.
Sentinel Prime MK4: healthy succeeded and injured succeeded.
Black Widow 2: healthy succeeded and injured succeeded.
Hawkeye: healthy succeeded and injured succeeded.
Nick Fury Jr: healthy succeeded and injured succeeded.
Emma Frost Normal: healthy succeeded and injured succeeded.
Emma Frost Diamond: healthy succeeded and injured succeeded.
Professor X: healthy succeeded and injured succeeded.
Psylocke: healthy succeeded and injured succeeded.
Star-Lord: healthy succeeded and injured succeeded.
Yondu: healthy succeeded and injured succeeded.

 Processing complete.
